#### Define notebook Parameters

Pass through th pl_worker

In [50]:
# Framework-compatible parameters with manual fallbacks.
import json
import logging
import uuid
from datetime import date

from pyspark.sql import functions as F



StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 72, Finished, Available, Finished, False)

In [51]:
%run nb_transformation_shared_functions

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 77, Finished, Available, Finished, True)

In [52]:
# Adds backticks to each part of a Fabric/Spark table name.
# Example: workspace.lakehouse.schema.table -> `workspace`.`lakehouse`.`schema`.`table`
def sql_identifier(name):
    return ".".join([f"`{part}`" for part in name.split(".")])


def lookup_function(table_lookup, key_mapping, return_column):
    # Wraps one key pair in a list so one-key and multi-key lookups use the same logic.
    if isinstance(key_mapping, tuple):
        key_mapping = [key_mapping]

    # Builds the SQL match condition between source columns and lookup table columns.
    where_clause = " AND ".join([
        f"{source_expression} = {lookup_expression}"
        for source_expression, lookup_expression in key_mapping
    ])
    
    # Looks up the requested column value and returns NULL when no match is found.
    return f"""
        (
            SELECT MAX(lkp.`{return_column}`)
            FROM {sql_identifier(table_lookup)} lkp
            WHERE {where_clause}
        )
    """

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 78, Finished, Available, Finished, False)

In [53]:
# Basic notebook runtime values
task_id = 50
task_name = "Load RMDD Address to Silver"
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False

# Manual watermark values
prev_wm = "1900-01-01 00:00:00"
curr_wm = "2026-06-30 00:00:00"

# Manual connection fallback
server_name = "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"
source_connection_settings = "{}"

# Source config
source_settings = json.dumps(
    {
        "source_name": "CAN_RMDD",
        "source_database": "CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e",
        "source_schema": "dbo",
        "table_config": [
            {
                "source_table": "Address",
                "target_table": "test_rmdd_address",
                "target_pk": ["address_id"],
                "is_incremental": 1,
                "incremental_column": "LastUpdatedDateGMT"
            }
        ]
    }
)

# Target config
target_settings = json.dumps(
    {
        "target_lakehouse": "lh_canada_global_mds",
        "target_schema": "silver_rmdd",
        "target_location": "Files/silver/test",
        "target_load_strategy": "overwrite"
    }
)

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 79, Finished, Available, Finished, False)

In [54]:
# Set up standard framework logging
setup_logging()
logger = logging.getLogger("LoadRMDDAddressToSilver")
logger.setLevel(logging.INFO)

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 80, Finished, Available, Finished, False)

In [55]:
# Accept either JSON strings from pipeline or already-parsed dicts for manual runs.
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings

source_connection_settings = (
    json.loads(source_connection_settings)
    if isinstance(source_connection_settings, str)
    else source_connection_settings
)

# Source configuration
source_name = source_settings.get("source_name")
source_database = source_settings.get("source_database") or source_settings.get("database_name")
source_schema = source_settings.get("source_schema")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse = target_settings.get("target_lakehouse")
target_schema = target_settings.get("target_schema")
target_location = target_settings.get("target_location")
target_load_strategy = target_settings.get("target_load_strategy", "merge-scd1")

# SQL connection configuration
server_name = (
    source_connection_settings.get("server_name")
    or source_connection_settings.get("jdbc_hostname")
    or source_connection_settings.get("server")
    or server_name
)

print(source_name)
print(source_database)
print(target_schema)
display(table_config)


StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 81, Finished, Available, Finished, False)

CAN_RMDD
CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e
silver_rmdd


SynapseWidget(Synapse.DataFrame, 1f8e0357-1399-4e6c-9ac8-a54c1be3011a)

In [56]:
# Build JDBC connection to source SQL database.
jdbc_url = (
    "jdbc:sqlserver://"
    f"{server_name}:1433;"
    f"database={source_database};"
    "encrypt=true;"
)

connection_properties = {
    "accessToken": mssparkutils.credentials.getToken("https://database.windows.net/"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print(jdbc_url)

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 82, Finished, Available, Finished, False)

jdbc:sqlserver://hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com:1433;database=CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e;encrypt=true;


In [57]:
# Read source tables.
source_dfs = {}

for cfg in table_config:
    source_table = cfg.get("source_table")
    source_table_schema = cfg.get("source_schema", source_schema)
    source_view = f"src_{to_snake_case(source_table)}"
    full_source_name = f"{source_table_schema}.{source_table}"

    source_query = f"""
    (
        SELECT *
        FROM {full_source_name}
    ) AS src
    """

    df_source = spark.read.jdbc(
        url=jdbc_url,
        table=source_query,
        properties=connection_properties
    )

    if limit_rows_for_debugging:
        df_source = df_source.limit(1000)

    df_source = remove_exact_duplicates(df_source)
    df_source.createOrReplaceTempView(source_view)
    source_dfs[source_table] = df_source

    print(f"Source preview for {source_view} ({full_source_name})")
    display(df_source.limit(3))

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 83, Finished, Available, Finished, False)

Source preview for src_address (dbo.Address)


SynapseWidget(Synapse.DataFrame, 0748f96c-72e8-4369-8969-2ac68940ca39)

In [58]:
# Map source views into target shape.
mapped_dfs = {}

for cfg in table_config:
    source_table = cfg.get("source_table")
    source_view = f"src_{to_snake_case(source_table)}"

    target_table = cfg.get("target_table")

    target_entity = to_snake_case(source_table)
    map_view = f"map_{target_entity}"

    watermark_column = cfg.get("incremental_column")
    is_incremental = cfg.get("is_incremental", 0)
    watermark_filter = "1 = 1"

    if is_incremental == 1 and watermark_column:
        watermark_filter = (
            f"s.{watermark_column} > '{prev_wm}' "
            f"AND s.{watermark_column} <= '{curr_wm}'"
        )

    if target_entity != "address":
        raise ValueError(f"No mapping defined for target entity: {target_entity}")

    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW {map_view} AS
    SELECT
        CAST(row_number() OVER (ORDER BY s.AddressID) AS BIGINT) AS address_key,
        CAST(s.AddressID AS INT) AS address_id,
        CAST(s.AddressTypeCode AS STRING) AS address_type_code,
        CAST(s.Address1 AS STRING) AS address_1,
        CAST(s.Address2 AS STRING) AS address_2,
        CAST(s.Address3 AS STRING) AS address_3,
        CAST(s.Address4 AS STRING) AS address_4,
        CAST(s.Address5 AS STRING) AS address_5,
        CAST(s.City AS STRING) AS city,
        CAST(s.CountryID AS INT) AS country_id,

        CAST({lookup_function(
        table_lookup="ws_canada_global_mds_dev.lh_canada_global_mds.stage_taxonomy.stage_taxonomy_curated_country",
        key_mapping=("s.CountryID","lkp.country_code_iso4"),
        return_column="country_code_iso2"
        )} AS STRING) AS country_code_iso2,

        CAST(s.State AS STRING) AS state,
        CAST(s.PostalCode AS STRING) AS zip_code,
        CAST(s.Address1 AS STRING) AS street_address_check,
        CAST(s.Active AS BOOLEAN) AS is_active,
        CAST(s.LastUpdatedDateGMT AS TIMESTAMP) AS last_updated_date_utc_at_source
    FROM {source_view} s
    WHERE {watermark_filter}
    """)

    mapped_dfs[target_table] = spark.table(map_view)

    print(f"Mapped preview for {map_view}")
    display(mapped_dfs[target_table].limit(3))

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 84, Finished, Available, Finished, False)

Mapped preview for map_address


SynapseWidget(Synapse.DataFrame, 12f6ca00-537d-4910-9533-9c5ad4430785)

In [59]:
spark.sql("SHOW TABLES IN lh_canada_global_mds.stage_taxonomy").show(truncate=False)

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 85, Finished, Available, Finished, False)

+------------------------------------------------------------+------------------------------------------------+-----------+
|namespace                                                   |tableName                                       |isTemporary|
+------------------------------------------------------------+------------------------------------------------+-----------+
|ws_canada_global_mds_dev.lh_canada_global_mds.stage_taxonomy|stage_taxonomy_curated_band_level               |false      |
|ws_canada_global_mds_dev.lh_canada_global_mds.stage_taxonomy|stage_taxonomy_curated_bill_type                |false      |
|ws_canada_global_mds_dev.lh_canada_global_mds.stage_taxonomy|stage_taxonomy_curated_client_sector            |false      |
|ws_canada_global_mds_dev.lh_canada_global_mds.stage_taxonomy|stage_taxonomy_curated_company                  |false      |
|ws_canada_global_mds_dev.lh_canada_global_mds.stage_taxonomy|stage_taxonomy_curated_country                  |false      |
|ws_cana

In [60]:
# Add metadata and hashes to mapped views.
target_dfs = {}

for cfg in table_config:
    source_table = cfg.get("source_table")
    target_table = cfg.get("target_table")
    target_pk = cfg.get("target_pk", [])

    target_prefix = f"{target_schema}_"

    target_entity = (
        target_table[len(target_prefix):]
        if target_table.startswith(target_prefix)
        else to_snake_case(target_table)
    )

    target_view = f"tgt_{target_entity}"
    source_file = f"{source_schema}.{source_table}"

    df_target = mapped_dfs[target_table]
    pk_expr_cols = [to_snake_case(col) for col in target_pk]

    df_target = df_target.withColumn(
        "_hashed_pk",
        F.md5(F.concat_ws("|", *[
            F.coalesce(F.col(col).cast("string"), F.lit("<NULL>"))
            for col in pk_expr_cols
        ]))
    )

    row_hash_cols = [
        col for col in df_target.columns
        if col not in [df_target.columns[0], "_hashed_pk"] + pk_expr_cols
    ]

    df_target = df_target.withColumn(
        "_hashed_row",
        F.md5(F.concat_ws("|", *[
            F.coalesce(F.col(col).cast("string"), F.lit("<NULL>"))
            for col in row_hash_cols
        ]))
    )

    df_target = add_metadata(
        df=df_target,
        ingest_date=str(date.today()),
        file_path=source_file,
        schema_name=source_name,
        table_name=target_table,
        lineage_id=lineage_id
    )

    df_target.createOrReplaceTempView(target_view)
    target_dfs[target_table] = df_target

    print(f"Target preview for {target_view}")
    display(df_target.limit(3))

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 86, Finished, Available, Finished, False)

Target preview for tgt_test_rmdd_address


SynapseWidget(Synapse.DataFrame, bdfdab60-447b-4152-949b-869c6c699d5a)

In [61]:
# Check duplicate hashed keys before merge.
for cfg in table_config:
    target_table = cfg.get("target_table")

    print(f"Checking duplicates for {target_table}")
    validate_duplicates(target_dfs[target_table], "_hashed_pk", 10)

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 87, Finished, Available, Finished, False)

Checking duplicates for test_rmdd_address
Total duplicate ['_hashed_pk'] groups: 0


In [62]:
# Merge final DataFrames to silver RMDD table.
load_results = []

for cfg in table_config:
    target_table = cfg.get("target_table")

    target_table_name = f"{target_schema}.{target_table}"
    target_location_path = f"{target_location}/{target_table}"
    df_target = target_dfs[target_table]

    result = load_to_target(
        df_target,
        target_table_name,
        target_load_strategy,
        target_location_path
    )

    load_results.append({
        "table": target_table_name,
        "rows_inserted": result["rows_inserted"],
        "rows_updated": result["rows_updated"],
        "rows_read": result["rows_read"],
        "rows_copied": result["rows_copied"]
    })

display(load_results)

StatementMeta(, 32de5891-c052-4d1e-b001-84e7b26bc8c8, 88, Finished, Available, Finished, False)

2026-07-02 23:49:41,411 UTC - INFO - Overwriting records in silver_rmdd.test_rmdd_address (LoadRMDDAddressToSilver)


SynapseWidget(Synapse.DataFrame, 6033f39f-f7b9-49a3-b8d5-11d3cae80cac)